In [7]:
# Lectura de librerias
print('Importando librerias')
import urllib.parse
import os
import pandas as pd
import numpy as np
import time
import pyarrow.parquet as pq
import pyarrow as pa
import polars as pl
import duckdb
from datetime import datetime
import psycopg2
from psycopg2 import OperationalError, sql
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.engine import URL
print("Librerias leidas")

Importando librerias
Librerias leidas


In [10]:
# Leyendo la data
print('Estableciendo conexion')
drivername="postgresql+psycopg2" # o "postgresql+psycopg"
username="postgres"
host="192.168.91.131"
port="5433"
database="cadm_j5004625301"
password=r"""Soloyo1234*-"""
url_nopassword = f"{drivername}://{username}@{host}:{port}/{database}"
engine = create_engine(
    url_nopassword,
    connect_args={
        "password": password,
        "client_encoding": "latin1"
    }
)
engine_2 = create_engine(
    url_nopassword,
    connect_args={
        "password": password,
        "client_encoding": "UTF8"
    }
)
print("Conexion establecida")

Estableciendo conexion
Conexion establecida


In [ ]:
# Ventas por dia
print('Extrayendo ventas')
query_cashea_orders = text("""
WITH ventas AS (
SELECT
 so.emission_date AS fecha,
 p.description AS producto,
 sd.amount AS und_vendidas,
CASE
 WHEN p.mark = 'X-BYTE' THEN 'X BYTE'
 WHEN p.mark = 'GO-PRO' THEN 'GO PRO'
 WHEN p.mark = '1-HORA' THEN '1 HORA'
 WHEN p.mark = 'PHILIPS-NORELCON' THEN 'PHILIPS NORELCON'
 WHEN p.mark = 'REALME ' THEN 'REALME'
 ELSE p.mark
END AS marca,
CASE
 WHEN so.user_code = 'ALMAYOR' THEN 'AL MAYOR'
 ELSE 'AL DETAL'
END AS grupo_venta,
CASE
 WHEN LEFT(UPPER(p.description),8) = 'TELEFONO' THEN 'TELEFONOS'
 WHEN d.father_department = '12' THEN 'ACCESORIOS'
 ELSE 'OTROS'
END AS grupo_productos
FROM sales_operation_details sd
INNER JOIN sales_operation so
ON sd.main_correlative = so.correlative
INNER JOIN products p
ON sd.code_product = p.code
INNER JOIN department d
ON d.code = p.department
WHERE
 so.client_name <> 'ANULADO'
 AND so.operation_type = 'BILL'
 AND so.user_code IN (
 'AB06', 'AB71', 'AB77', 'AB77AC', 'B214', 'B407', 'L111', 'L160', 'L160AA', 'L210L', 'L210M',
 'L210Y', 'L227N', 'L227O', 'L229', 'L230', 'L231', 'L246', 'L250', 'L251', 'L337Q', 'L337R',
 'L337Z', 'L249', 'DELIVERY', 'ALMAYOR'
 )
),
time_series AS (
SELECT
 fecha, grupo_venta, grupo_productos, marca,
 SUM(und_vendidas) AS und_vendidas
FROM ventas
WHERE
fecha BETWEEN '2026-01-01' AND (CURRENT_DATE - INTERVAL '1 day')
AND marca NOT IN ('', '-')
AND grupo_productos <> 'OTROS'
GROUP BY fecha, grupo_venta, grupo_productos, marca
),
top_declared AS (
SELECT
 producto, marca, grupo_venta, grupo_productos,
 SUM(und_vendidas) AS und_vendidas
FROM ventas
WHERE
fecha BETWEEN '2026-01-01' AND (CURRENT_DATE - INTERVAL '1 day')
AND marca NOT IN ('', '-')
AND grupo_productos <> 'OTROS'
GROUP BY producto, marca, grupo_venta, grupo_productos
),
top_selected AS (
SELECT
 producto,
 marca,
 grupo_venta,
 grupo_productos,
 und_vendidas,
 DENSE_RANK() OVER(PARTITION BY grupo_venta, grupo_productos, marca ORDER BY und_vendidas DESC) AS top
FROM top_declared
)
SELECT *
FROM time_series;
""")
chunk_size = 5000
output_file_seller = r'C:\Users\POWERBI\Desktop\Entornos\kpi_monthly\kpi_reports\time_series_orders.csv'
first_chunk = True

with engine.connect() as conn:
    conn = conn.execution_options(
        isolation_level="AUTOCOMMIT",
        postgresql_readonly=True
    )
    chunks = pd.read_sql_query(query_cashea_orders, con=conn, chunksize=chunk_size)
    
    for i, chunk in enumerate(chunks):
        print(f"Procesando bloque {i + 1}...")
        
        # Escribir CSV en modo streaming
        if first_chunk:
            # Primer bloque: escribir con cabeceras
            chunk.to_csv(output_file_seller, index=False, mode='w', encoding='utf-8-sig')
            first_chunk = False
        else:
            # Bloques siguientes: escribir sin cabeceras (modo append)
            chunk.to_csv(output_file_seller, index=False, mode='a', header=False, encoding='utf-8-sig')
print("Ventas exportadas")

In [ ]:
# Productos mas vendidos por marca
print('Extrayendo productos mas vendidos')
query_cashea_orders = text("""
WITH ventas AS (
SELECT
 so.emission_date AS fecha,
 p.description AS producto,
 sd.amount AS und_vendidas,
CASE
 WHEN p.mark = 'X-BYTE' THEN 'X BYTE'
 WHEN p.mark = 'GO-PRO' THEN 'GO PRO'
 WHEN p.mark = '1-HORA' THEN '1 HORA'
 WHEN p.mark = 'PHILIPS-NORELCON' THEN 'PHILIPS NORELCON'
 WHEN p.mark = 'REALME ' THEN 'REALME'
 ELSE p.mark
END AS marca,
CASE
 WHEN so.user_code = 'ALMAYOR' THEN 'AL MAYOR'
 ELSE 'AL DETAL'
END AS grupo_venta,
CASE
 WHEN LEFT(UPPER(p.description),8) = 'TELEFONO' THEN 'TELEFONOS'
 WHEN d.father_department = '12' THEN 'ACCESORIOS'
 ELSE 'OTROS'
END AS grupo_productos
FROM sales_operation_details sd
INNER JOIN sales_operation so
ON sd.main_correlative = so.correlative
INNER JOIN products p
ON sd.code_product = p.code
INNER JOIN department d
ON d.code = p.department
WHERE
 so.client_name <> 'ANULADO'
 AND so.operation_type = 'BILL'
 AND so.user_code IN (
 'AB06', 'AB71', 'AB77', 'AB77AC', 'B214', 'B407', 'L111', 'L160', 'L160AA', 'L210L', 'L210M',
 'L210Y', 'L227N', 'L227O', 'L229', 'L230', 'L231', 'L246', 'L250', 'L251', 'L337Q', 'L337R',
 'L337Z', 'L249', 'DELIVERY', 'ALMAYOR'
 )
),
time_series AS (
SELECT
 fecha, grupo_venta, grupo_productos, marca,
 SUM(und_vendidas) AS und_vendidas
FROM ventas
WHERE
fecha BETWEEN '2026-01-01' AND (CURRENT_DATE - INTERVAL '1 day')
AND marca NOT IN ('', '-')
AND grupo_productos <> 'OTROS'
GROUP BY fecha, grupo_venta, grupo_productos, marca
),
top_declared AS (
SELECT
 EXTRACT(MONTH FROM fecha) AS mes, producto, marca, grupo_venta, grupo_productos,
 SUM(und_vendidas) AS und_vendidas
FROM ventas
WHERE
fecha BETWEEN '2026-01-01' AND (CURRENT_DATE - INTERVAL '1 day')
AND marca NOT IN ('', '-')
AND grupo_productos <> 'OTROS'
GROUP BY EXTRACT(MONTH FROM fecha), producto, marca, grupo_venta, grupo_productos
),
top_selected AS (
SELECT
 mes,
 producto,
 marca,
 grupo_venta,
 grupo_productos,
 und_vendidas,
 DENSE_RANK() OVER(PARTITION BY mes, grupo_venta, grupo_productos, marca ORDER BY und_vendidas DESC) AS top
FROM top_declared
)
SELECT *
FROM top_selected
WHERE top <= 10;
""")
chunk_size = 5000
output_file_seller = r'C:\Users\POWERBI\Desktop\Entornos\kpi_monthly\kpi_reports\top_products_orders.csv'
first_chunk = True
with engine.connect() as conn:
    conn = conn.execution_options(
        isolation_level="AUTOCOMMIT",
        postgresql_readonly=True
    )
    chunks = pd.read_sql_query(query_cashea_orders, con=conn, chunksize=chunk_size)
    
    for i, chunk in enumerate(chunks):
        print(f"Procesando bloque {i + 1}...")
        
        # Escribir CSV en modo streaming
        if first_chunk:
            # Primer bloque: escribir con cabeceras
            chunk.to_csv(output_file_seller, index=False, mode='w', encoding='utf-8-sig')
            first_chunk = False
        else:
            # Bloques siguientes: escribir sin cabeceras (modo append)
            chunk.to_csv(output_file_seller, index=False, mode='a', header=False, encoding='utf-8-sig')
print("Productos mas vendidos exportados")

In [ ]:
# Ventas por precio de venta
print('Extrayendo ventas por precio de venta')
query_pricing_cashea = text("""
WITH ventas AS (
SELECT
 so.emission_date AS fecha,
 p.description AS producto,
 sd.amount AS und_vendidas,
CASE
 WHEN p.mark = 'X-BYTE' THEN 'X BYTE'
 WHEN p.mark = 'GO-PRO' THEN 'GO PRO'
 WHEN p.mark = '1-HORA' THEN '1 HORA'
 WHEN p.mark = 'PHILIPS-NORELCON' THEN 'PHILIPS NORELCON'
 WHEN p.mark = 'REALME ' THEN 'REALME'
 ELSE p.mark
END AS marca,
CASE
 WHEN so.user_code = 'ALMAYOR' THEN 'AL MAYOR'
 ELSE 'AL DETAL'
END AS grupo_venta,
CASE
 WHEN LEFT(UPPER(p.description),8) = 'TELEFONO' THEN 'TELEFONOS'
 WHEN d.father_department = '12' THEN 'ACCESORIOS'
 ELSE 'OTROS'
END AS grupo_productos
FROM sales_operation_details sd
INNER JOIN sales_operation so
ON sd.main_correlative = so.correlative
INNER JOIN products p
ON sd.code_product = p.code
INNER JOIN department d
ON d.code = p.department
WHERE
 so.client_name <> 'ANULADO'
 AND so.operation_type = 'BILL'
 AND so.user_code IN (
 'AB06', 'AB71', 'AB77', 'AB77AC', 'B214', 'B407', 'L111', 'L160', 'L160AA', 'L210L', 'L210M',
 'L210Y', 'L227N', 'L227O', 'L229', 'L230', 'L231', 'L246', 'L250', 'L251', 'L337Q', 'L337R',
 'L337Z', 'L249', 'DELIVERY', 'ALMAYOR'
 )
),
time_series AS (
SELECT
 fecha, grupo_venta, grupo_productos, marca,
 SUM(und_vendidas) AS und_vendidas
FROM ventas
WHERE
fecha BETWEEN '2026-01-01' AND (CURRENT_DATE - INTERVAL '1 day')
AND marca NOT IN ('', '-')
AND grupo_productos <> 'OTROS'
GROUP BY fecha, grupo_venta, grupo_productos, marca
),
top_declared AS (
SELECT
 producto, marca, grupo_venta, grupo_productos,
 SUM(und_vendidas) AS und_vendidas
FROM ventas
WHERE
fecha BETWEEN '2026-01-01' AND (CURRENT_DATE - INTERVAL '1 day')
AND marca NOT IN ('', '-')
AND grupo_productos <> 'OTROS'
GROUP BY producto, marca, grupo_venta, grupo_productos
),
top_selected AS (
SELECT
 producto,
 marca,
 grupo_venta,
 grupo_productos,
 und_vendidas,
 DENSE_RANK() OVER(PARTITION BY grupo_venta, grupo_productos, marca ORDER BY und_vendidas DESC) AS top
FROM top_declared
)
SELECT *
FROM time_series;
""")
chunk_size = 5000
output_file_seller = r'C:\Users\POWERBI\Desktop\Entornos\kpi_monthly\kpi_reports\pricing_cashea.csv'
first_chunk = True

with engine.connect() as conn:
    conn = conn.execution_options(
        isolation_level="AUTOCOMMIT",
        postgresql_readonly=True
    )
    chunks = pd.read_sql_query(query_pricing_cashea, con=conn, chunksize=chunk_size)
    
    for i, chunk in enumerate(chunks):
        print(f"Procesando bloque {i + 1}...")
        
        # Escribir CSV en modo streaming
        if first_chunk:
            # Primer bloque: escribir con cabeceras
            chunk.to_csv(output_file_seller, index=False, mode='w', encoding='utf-8-sig')
            first_chunk = False
        else:
            # Bloques siguientes: escribir sin cabeceras (modo append)
            chunk.to_csv(output_file_seller, index=False, mode='a', header=False, encoding='utf-8-sig')
print("Ventas por precio exportadas")